In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install pandas numpy tqdm

In [ ]:
import os
import re
import json
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Configuration
MODEL_NAME = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"
DATASET_PATH = "/kaggle/input/datasets/mhariyan/eps-forecast-dataset/final_verified_dataset (1).jsonl"

In [ ]:
# 4-bit Config for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # T4 preferred dtype
    bnb_4bit_use_double_quant=True,
)

print(f"Loading Base Model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto", # Automatically uses T4 x 2
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.eval()

In [ ]:
# 1. CRITICAL CHECK: Verify Kaggle sees the GPUs
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU not detected! Go to Settings -> Accelerator -> T4 x 2 and restart.")
else:
    print(f"✅ CUDA detected. Found {torch.cuda.device_count()} GPUs.")

MODEL_NAME = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"

# 2. Optimized Config for T4 (uses float16, as T4 doesn't support bfloat16 well)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=True,
)

print("Loading model onto dual GPUs...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto", # This splits the model across both T4s 
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def parse_forecast(text):
    # Remove thinking process to avoid parsing numbers inside the chain of thought
    clean_text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # Look for "Forecast: [number]" or "EPS: [number]" in the final answer
    patterns = [
        r"(?i)forecast\s*[:=]\s*(-?\d+\.?\d*)",
        r"(?i)eps\s*[:=]\s*(-?\d+\.?\d*)"
    ]
    for pattern in patterns:
        match = re.search(pattern, clean_text)
        if match:
            return float(match.group(1))
    return None

def get_prev_eps(input_text):
    matches = re.findall(r'EPS:\s*(-?\d+(?:\.\d+)?)', input_text)
    return float(matches[-1]) if matches else None

# Load Data
samples = []
with open(DATASET_PATH, 'r') as f:
    for line in f:
        samples.append(json.loads(line))

results = []
print(f"Starting Inference on {len(samples)} samples...")

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
<think>
"""

for sample in tqdm(samples):
    prompt = ALPACA_PROMPT.format(instruction=sample['instruction'], input=sample['input'])
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False, # Greedy for ablation consistency
            temperature=0,
            use_cache=True
        )
    
    # Decode only the new tokens
    full_output = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    response = full_output.split("### Response:")[-1]
    
    pred_eps = parse_forecast(response)
    actual_eps = parse_forecast(sample['output']) # Extracting GT from JSONL
    prev_eps = get_prev_eps(sample['input'])
    
    if pred_eps is not None and actual_eps is not None:
        results.append({
            "actual": actual_eps,
            "predicted": pred_eps,
            "prev_eps": prev_eps
        })

# Save interim results
df = pd.DataFrame(results)
df.to_csv("ablation_results.csv", index=False)

In [ ]:
def calculate_metrics(df):
    y_true = df['actual'].values
    y_pred = df['predicted'].values
    
    # Basic Errors
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    
    # Percentage Errors (handling zero actuals with epsilon)
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    
    # MAAPE (Arctangent Absolute Percentage Error)
    maape = np.mean(np.arctan(np.abs((y_true - y_pred) / (y_true + epsilon))))
    
    # Directional Accuracy (DA)
    # Checks if predicted change matches actual change direction from previous year
    df_da = df.dropna(subset=['prev_eps'])
    actual_dir = np.sign(df_da['actual'] - df_da['prev_eps'])
    pred_dir = np.sign(df_da['predicted'] - df_da['prev_eps'])
    da = (actual_dir == pred_dir).mean() * 100

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE%": mape,
        "MdAPE%": mdape,
        "MAAPE": maape,
        "DA%": da
    }

metrics = calculate_metrics(df)

print("\n" + "="*30)
print("   ABLATION STUDY RESULTS")
print("="*30)
for k, v in metrics.items():
    print(f"{k:<10}: {v:.4f}")
print("="*30)